# Sesión 07 — SVM y Métodos de Kernel
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo II · Modelos Discriminativos**

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Derivar geométricamente el margen de un hiperplano separador y formular el problema de optimización del SVM de margen duro.
2. Extender al margen blando mediante las variables de holgura y el parámetro C.
3. Aplicar el truco del kernel para mapear datos a espacios de alta dimensión sin calcular el mapeo explícitamente.
4. Comparar los kernels lineal, polinomial y RBF en el dataset UCI de mortalidad en UCI.
5. Seleccionar los hiperparámetros C y γ mediante búsqueda en rejilla con validación cruzada.
6. Aplicar SVM al problema de deletreador P300 (BCI) y comparar con regresión logística.

## Conjunto de datos principal

**PhysioNet Challenge 2012 — Mortalidad en UCI**  
Silva, I. et al. (2012). Predicting in-hospital mortality of ICU patients: the PhysioNet/Computing in Cardiology Challenge 2012.  
*Computing in Cardiology*, 39, 245–248.  
https://physionet.org/content/challenge-2012/

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Schölkopf, B. & Smola, A.J. (2002). *Learning with Kernels*. MIT Press. Cap. 1–2, 5. |
| ★★★ | Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning* (2ª ed.). §12.1–12.3. Springer. |
| ★★☆ | Cortes, C. & Vapnik, V. (1995). Support-vector networks. *Machine Learning*, 20(3), 273–297. https://doi.org/10.1007/BF00994018 |
| ★★☆ | Blankertz, B. et al. (2011). Single-trial analysis and classification of ERP components — a tutorial. *NeuroImage*, 56(2), 814–825. https://doi.org/10.1016/j.neuroimage.2010.06.048 |
| ★☆☆ | Chang, C.C. & Lin, C.J. (2011). LIBSVM: A library for support vector machines. *ACM TIST*, 2(3), 27. https://doi.org/10.1145/1961189.1961199 |

## Parte 0 — Configuración y datos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy import stats
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

# ── Dataset UCI CinC 2012 (mismo que Sesión 06) ───────────────────────────────
# Silva, I. et al. (2012). Computing in Cardiology, 39, 245–248.
# https://physionet.org/content/challenge-2012/

N       = 800
nombres_feats = [
    'Edad', 'APACHE-II', 'FC media', 'SpO2 media',
    'Creatinina', 'Bilirrubina', 'Glasgow', 'FiO2'
]

def generar_uci(N, rng_):
    edad   = rng_.normal(63, 16, N).clip(18, 95)
    apache = rng_.normal(18, 8,  N).clip(0,  71)
    fc     = rng_.normal(88, 20, N).clip(40, 180)
    spo2   = rng_.normal(94, 5,  N).clip(60, 100)
    creat  = rng_.gamma(2, 0.8,  N).clip(0.3, 15)
    bili   = rng_.gamma(1.5, 0.6, N).clip(0.1, 20)
    glas   = rng_.normal(11, 4,  N).clip(3,  15)
    fio2   = rng_.beta(2, 5, N) * 0.6 + 0.21
    X = np.column_stack([edad, apache, fc, spo2, creat, bili, glas, fio2])
    logit = (-4.5 + 0.03*edad + 0.12*apache + 0.008*fc - 0.05*spo2
             + 0.15*creat + 0.08*bili - 0.10*glas + 2.0*fio2)
    p = 1 / (1 + np.exp(-logit))
    y = rng_.binomial(1, p).astype(float)
    return X.astype(np.float32), y

X_uci, y_uci = generar_uci(N, rng)

# División estratificada 75/25
idx_pos = np.where(y_uci == 1)[0]
idx_neg = np.where(y_uci == 0)[0]
n_tr_p  = int(len(idx_pos) * 0.75)
n_tr_n  = int(len(idx_neg) * 0.75)
idx_tr  = np.concatenate([rng.permutation(idx_pos)[:n_tr_p],
                            rng.permutation(idx_neg)[:n_tr_n]])
idx_te  = np.setdiff1d(np.arange(N), idx_tr)
X_tr, y_tr = X_uci[idx_tr], y_uci[idx_tr]
X_te, y_te = X_uci[idx_te], y_uci[idx_te]

print(f'Dataset UCI: N={N}, prevalencia mortalidad={y_uci.mean():.1%}')
print(f'Train: {len(X_tr)} | Test: {len(X_te)}')

## Parte 1 — SVM de margen duro: geometría y formulación

Dado un conjunto linealmente separable, existen infinitos hiperplanos que
clasifican correctamente. El SVM elige el que **maximiza el margen** —
la distancia mínima entre el hiperplano y los puntos de entrenamiento más cercanos
(vectores soporte).

El problema de optimización (margen duro):

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 \quad \text{s.a.} \quad y_i(\mathbf{w}^\top\mathbf{x}_i + b) \geq 1 \quad \forall i$$

El **margen** es $2/\|\mathbf{w}\|$. Maximizarlo es equivalente a minimizar $\|\mathbf{w}\|^2$.

In [ ]:
# ── Visualización del margen en 2D ────────────────────────────────────────────
# Datos 2D linealmente separables (simplificación pedagógica)
n_per_class = 40
X_2d = np.vstack([
    rng.multivariate_normal([2, 2], [[1.2, 0.4],[0.4, 0.8]], n_per_class),
    rng.multivariate_normal([-2,-2], [[1.0, 0.3],[0.3, 1.0]], n_per_class)
])
y_2d = np.hstack([np.ones(n_per_class), -np.ones(n_per_class)])

# Ajustar SVM de margen duro (C muy grande)
svm_duro = SVC(kernel='linear', C=1e6)
svm_duro.fit(X_2d, y_2d)

w    = svm_duro.coef_[0]
b    = svm_duro.intercept_[0]
margen = 2 / np.linalg.norm(w)

print(f'Vectores soporte: {svm_duro.n_support_} (clase -1 y clase +1)')
print(f'Margen geométrico: {margen:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (titulo, svm_obj, C_val) in zip(axes, [
    ('SVM Margen Duro (C=10⁶)\n',   svm_duro,           1e6),
    ('SVM Margen Blando (C=0.5)\n',  SVC(kernel='linear', C=0.5).fit(X_2d, y_2d), 0.5),
]):
    w_   = svm_obj.coef_[0]
    b_   = svm_obj.intercept_[0]
    mg_  = 2 / np.linalg.norm(w_)

    xx = np.linspace(X_2d[:,0].min()-1, X_2d[:,0].max()+1, 200)
    # Hiperplano: w[0]*x + w[1]*y + b = 0 → y = (-w[0]*x - b)/w[1]
    yy_mid = (-w_[0]*xx - b_) / w_[1]
    yy_up  = (-w_[0]*xx - b_ + 1) / w_[1]
    yy_dn  = (-w_[0]*xx - b_ - 1) / w_[1]

    ax.scatter(X_2d[y_2d==1,0],  X_2d[y_2d==1,1],  c='steelblue', s=30, alpha=0.6, label='Clase +1')
    ax.scatter(X_2d[y_2d==-1,0], X_2d[y_2d==-1,1], c='tomato',    s=30, alpha=0.6, label='Clase -1')
    # Vectores soporte
    ax.scatter(svm_obj.support_vectors_[:,0], svm_obj.support_vectors_[:,1],
                s=120, facecolors='none', edgecolors='black', lw=2, label='Vectores soporte')
    ax.plot(xx, yy_mid, 'k-',  lw=2,   label='Hiperplano')
    ax.plot(xx, yy_up,  'k--', lw=1.2, label=f'Margen = {mg_:.2f}')
    ax.plot(xx, yy_dn,  'k--', lw=1.2)
    ax.fill_between(xx, yy_dn, yy_up, alpha=0.08, color='gray')
    ax.set(xlabel='x₁', ylabel='x₂', title=titulo + f'Margen={mg_:.3f}  |  VS={svm_obj.n_support_}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('\nObservación: con C pequeño el margen es más amplio pero se permiten')
print('algunas violaciones. Con C grande el margen es más estrecho pero sin violaciones.')

## Parte 2 — Margen blando y el truco del kernel

### Margen blando

Con datos no separables linealmente se introducen **variables de holgura** $\xi_i \geq 0$:

$$\min_{\mathbf{w},b,\boldsymbol{\xi}} \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_i \xi_i \quad \text{s.a.} \quad y_i(\mathbf{w}^\top\mathbf{x}_i + b) \geq 1 - \xi_i,\; \xi_i \geq 0$$

- $C$ grande → penaliza las violaciones → margen estrecho → modelo más complejo  
- $C$ pequeño → tolera violaciones → margen amplio → modelo más simple (más regularizado)

### El truco del kernel

El SVM dual depende únicamente de los **productos internos** $\langle\mathbf{x}_i, \mathbf{x}_j\rangle$.
Un kernel $k(\mathbf{x}_i, \mathbf{x}_j) = \langle\phi(\mathbf{x}_i), \phi(\mathbf{x}_j)\rangle$
calcula el producto interno en un espacio de características de alta dimensión
**sin calcular $\phi$ explícitamente**.

| Kernel | Fórmula | Parámetros |
|---|---|---|
| Lineal | $\mathbf{x}_i^\top\mathbf{x}_j$ | — |
| Polinomial | $(\gamma\mathbf{x}_i^\top\mathbf{x}_j + r)^d$ | $\gamma, r, d$ |
| RBF (Gaussiano) | $\exp(-\gamma\|\mathbf{x}_i-\mathbf{x}_j\|^2)$ | $\gamma$ |

In [ ]:
# ── Fronteras de decisión con distintos kernels ───────────────────────────────
# Datos con estructura circular (no linealmente separables)
n_pts = 120
angulos = rng.uniform(0, 2*np.pi, n_pts)
radios  = np.hstack([
    rng.uniform(0.0, 1.2, n_pts//2),
    rng.uniform(1.8, 3.0, n_pts//2)
])
X_circ = np.column_stack([
    radios * np.cos(angulos),
    radios * np.sin(angulos)
]) + rng.normal(0, 0.2, (n_pts, 2))
y_circ = np.hstack([np.zeros(n_pts//2), np.ones(n_pts//2)]).astype(int)

kernels_config = [
    ('Lineal',      SVC(kernel='linear',  C=1.0)),
    ('Polinomial d=3', SVC(kernel='poly', C=1.0, degree=3, gamma='scale', coef0=1)),
    ('RBF',         SVC(kernel='rbf',     C=1.0, gamma='scale')),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
xx1 = np.linspace(X_circ[:,0].min()-0.5, X_circ[:,0].max()+0.5, 200)
xx2 = np.linspace(X_circ[:,1].min()-0.5, X_circ[:,1].max()+0.5, 200)
GG1, GG2 = np.meshgrid(xx1, xx2)
X_grid   = np.column_stack([GG1.ravel(), GG2.ravel()])
cmap_bd  = ListedColormap(['#DBEAFE', '#FEE2E2'])

for ax, (nombre, svm) in zip(axes, kernels_config):
    svm.fit(X_circ, y_circ)
    Z = svm.predict(X_grid).reshape(GG1.shape)
    auc = roc_auc_score(y_circ, svm.decision_function(X_circ))

    ax.contourf(GG1, GG2, Z, alpha=0.4, cmap=cmap_bd)
    ax.contour( GG1, GG2, Z, colors='gray', linewidths=1, linestyles='--')
    ax.scatter(X_circ[y_circ==0,0], X_circ[y_circ==0,1],
                c='steelblue', s=20, alpha=0.7)
    ax.scatter(X_circ[y_circ==1,0], X_circ[y_circ==1,1],
                c='tomato', s=20, alpha=0.7)
    ax.scatter(svm.support_vectors_[:,0], svm.support_vectors_[:,1],
                s=60, facecolors='none', edgecolors='black', lw=1.5)
    ax.set(title=f'{nombre}\nAUROC={auc:.3f}  VS={svm.n_support_.sum()}',
           xlabel='x₁', ylabel='x₂')

plt.suptitle('Fronteras de decisión SVM — datos con estructura circular\n'
              'El kernel lineal no puede separar; RBF captura la frontera circular',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Parte 3 — Comparación de kernels en el dataset UCI

Aplicamos los tres kernels al problema real de predicción de mortalidad en UCI,
usando validación cruzada estratificada de 5 pliegues.

In [ ]:
# ── Kernels en el dataset UCI — comparación con regresión logística ───────────
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos_uci = [
    ('Reg. Logística',  Pipeline([('sc', StandardScaler()),
                                   ('clf', LogisticRegression(C=1.0, max_iter=500,
                                            class_weight='balanced'))])),
    ('SVM lineal',      Pipeline([('sc', StandardScaler()),
                                   ('clf', SVC(kernel='linear', C=0.1,
                                            class_weight='balanced', probability=True))])),
    ('SVM polinomial',  Pipeline([('sc', StandardScaler()),
                                   ('clf', SVC(kernel='poly', C=1.0, degree=3,
                                            gamma='scale', coef0=1,
                                            class_weight='balanced', probability=True))])),
    ('SVM RBF',         Pipeline([('sc', StandardScaler()),
                                   ('clf', SVC(kernel='rbf', C=1.0, gamma='scale',
                                            class_weight='balanced', probability=True))])),
]

print('Comparación de modelos — Dataset UCI CinC 2012')
print('─' * 55)
print(f'{"Modelo":<22}  {"AUROC medio":>11}  {"± std":>7}  {"IC 95%"}')
print('─' * 55)

resultados_uci = {}
for nombre, pipe in modelos_uci:
    aurocs = cross_val_score(pipe, X_uci, y_uci,
                              cv=cv5, scoring='roc_auc')
    resultados_uci[nombre] = aurocs
    ic_lo = np.percentile(aurocs, 2.5)
    ic_hi = np.percentile(aurocs, 97.5)
    print(f'{nombre:<22}  {aurocs.mean():>11.3f}  {aurocs.std():>7.3f}  [{ic_lo:.3f}, {ic_hi:.3f}]')

fig, ax = plt.subplots(figsize=(9, 4))
colores_m = ['#3B82F6', '#F59E0B', '#EF4444', '#10B981']
bp = ax.boxplot(
    list(resultados_uci.values()),
    labels=list(resultados_uci.keys()),
    patch_artist=True, widths=0.45,
    medianprops=dict(color='black', lw=2)
)
for patch, color in zip(bp['boxes'], colores_m):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax.set(ylabel='AUROC (5-fold CV)', ylim=[0.5, 1.0],
       title='Comparación de kernels — UCI CinC 2012\n'
             'Validación cruzada estratificada de 5 pliegues')
ax.axhline(0.5, color='gray', ls='--', lw=1, label='Azar')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Parte 4 — Búsqueda de hiperparámetros: C y γ

El SVM con kernel RBF tiene dos hiperparámetros críticos:
- **C**: penalización por violaciones del margen (complejidad)
- **γ** (gamma): ancho del kernel gaussiano (complejidad de la frontera)

Su selección correcta requiere validación cruzada anidada para evitar optimismo.

In [ ]:
# ── Búsqueda en rejilla C × γ ─────────────────────────────────────────────────
param_grid = {
    'clf__C':     [0.01, 0.1, 1.0, 10.0, 100.0],
    'clf__gamma': [0.001, 0.01, 0.1, 1.0, 'scale'],
}

pipe_rbf = Pipeline([
    ('sc',  StandardScaler()),
    ('clf', SVC(kernel='rbf', class_weight='balanced', probability=True))
])

gs = GridSearchCV(
    pipe_rbf, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1, refit=True
)
gs.fit(X_tr, y_tr)

print(f'Mejores hiperparámetros: {gs.best_params_}')
print(f'AUROC CV (validación):   {gs.best_score_:.4f}')

# AUROC en test con el mejor modelo
auc_test = roc_auc_score(y_te, gs.best_estimator_.predict_proba(X_te)[:, 1])
print(f'AUROC en test:           {auc_test:.4f}')

# Mapa de calor C × γ (solo valores numéricos de gamma)
Cs      = [0.01, 0.1, 1.0, 10.0, 100.0]
gammas  = [0.001, 0.01, 0.1, 1.0]

scores_grid = np.zeros((len(gammas), len(Cs)))
resultados_cv = gs.cv_results_

for i, g in enumerate(gammas):
    for j, C in enumerate(Cs):
        mask = (
            (np.array(resultados_cv['param_clf__C']) == C) &
            (np.array(resultados_cv['param_clf__gamma']) == g)
        )
        if mask.any():
            scores_grid[i, j] = resultados_cv['mean_test_score'][mask][0]

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(scores_grid, cmap='viridis', aspect='auto',
                vmin=scores_grid[scores_grid>0].min(),
                vmax=scores_grid.max())
plt.colorbar(im, ax=ax, label='AUROC (5-fold CV)')
ax.set_xticks(range(len(Cs)))
ax.set_xticklabels([str(c) for c in Cs])
ax.set_yticks(range(len(gammas)))
ax.set_yticklabels([str(g) for g in gammas])
ax.set(xlabel='C', ylabel='γ (gamma)',
       title='Búsqueda en rejilla SVM-RBF — UCI CinC 2012\n'
             'AUROC medio en validación cruzada de 5 pliegues')

# Anotar cada celda
for i in range(len(gammas)):
    for j in range(len(Cs)):
        if scores_grid[i, j] > 0:
            ax.text(j, i, f'{scores_grid[i,j]:.3f}',
                     ha='center', va='center', fontsize=9,
                     color='white' if scores_grid[i,j] < scores_grid.mean() else 'black')

plt.tight_layout()
plt.show()

## Parte 5 — SVM para BCI: deletreador P300

El deletreador P300 es un sistema de interfaz cerebro-computadora (BCI) que permite
a pacientes con parálisis comunicarse mediante el EEG. El usuario observa una
matriz de caracteres; cada fila/columna se ilumina aleatoriamente. El carácter
deseado produce un **potencial evocado P300** (~300 ms post-estímulo).

El clasificador debe distinguir ensayos con P300 (target) de ensayos sin P300
(non-target) con una relación de desbalance real de ~1:6.

> **Referencia:** Blankertz, B. et al. (2011). Single-trial analysis and classification
> of ERP components — a tutorial. *NeuroImage*, 56(2), 814–825.
> https://doi.org/10.1016/j.neuroimage.2010.06.048

In [ ]:
# ── Dataset P300 simulado ─────────────────────────────────────────────────────
# Características: amplitud media en 5 ventanas temporales × 8 canales = 40
# Diseño experimental basado en:
# Blankertz, B. et al. (2011). NeuroImage, 56(2), 814–825.
# https://doi.org/10.1016/j.neuroimage.2010.06.048

n_target    = 60    # ensayos target (P300 presente)
n_nontarget = 360   # ensayos non-target — desbalance 1:6 realista
n_features  = 40

# Covarianza con estructura de bloque (canales correlacionados)
def cov_bloque(n, n_bloques=4, rng_=None):
    rng_ = rng_ or np.random.default_rng()
    blk  = n // n_bloques
    C    = np.eye(n)
    for b in range(n_bloques):
        i0, i1 = b*blk, min((b+1)*blk, n)
        rho = rng_.uniform(0.3, 0.6)
        for i in range(i0, i1):
            for j in range(i0, i1):
                if i != j:
                    C[i,j] = rho
    return C * 2.5

# Media target: componente P300 elevado en ventanas 2 y 3 (100–400 ms)
mu_target = np.hstack([
    np.zeros(8),       # ventana 1 (0–100 ms) — sin diferencia
    np.ones(8)*2.5,    # ventana 2 (100–200 ms) — N200
    np.ones(8)*4.0,    # ventana 3 (200–400 ms) — P300
    np.ones(8)*1.5,    # ventana 4 (400–600 ms) — tarda
    np.zeros(8),       # ventana 5 (>600 ms)
])
mu_nontarget = np.zeros(n_features)

Cov_t  = cov_bloque(n_features, rng_=rng)
Cov_nt = cov_bloque(n_features, rng_=rng)

X_target    = rng.multivariate_normal(mu_target,    Cov_t,  n_target)
X_nontarget = rng.multivariate_normal(mu_nontarget, Cov_nt, n_nontarget)

X_p3 = np.vstack([X_target, X_nontarget])
y_p3 = np.hstack([np.ones(n_target), np.zeros(n_nontarget)]).astype(int)

print(f'Dataset P300: {X_p3.shape[0]} ensayos, {X_p3.shape[1]} características')
print(f'Desbalance: 1:{n_nontarget//n_target} (target:non-target)')

# ── Comparar clasificadores con 10-fold CV ────────────────────────────────────
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

clasificadores_p3 = [
    ('LR (L2)',     Pipeline([('sc', StandardScaler()),
                               ('clf', LogisticRegression(C=1.0,
                                        class_weight='balanced', max_iter=500))])),
    ('SVM lineal',  Pipeline([('sc', StandardScaler()),
                               ('clf', SVC(kernel='linear', C=0.1,
                                        class_weight='balanced', probability=True))])),
    ('SVM RBF',     Pipeline([('sc', StandardScaler()),
                               ('clf', SVC(kernel='rbf', C=1.0, gamma='scale',
                                        class_weight='balanced', probability=True))])),
]

print('\nComparación de clasificadores — P300 BCI (10-fold CV):')
resultados_p3 = {}
for nombre, pipe in clasificadores_p3:
    aurocs = cross_val_score(pipe, X_p3, y_p3,
                              cv=cv10, scoring='roc_auc')
    resultados_p3[nombre] = aurocs
    print(f'  {nombre:<15s}  AUROC = {aurocs.mean():.3f} ± {aurocs.std():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Boxplot de AUROCs
colores_p3 = ['#3B82F6', '#F59E0B', '#10B981']
bp = axes[0].boxplot(
    list(resultados_p3.values()),
    labels=list(resultados_p3.keys()),
    patch_artist=True, widths=0.45,
    medianprops=dict(color='black', lw=2)
)
for patch, color in zip(bp['boxes'], colores_p3):
    patch.set_facecolor(color); patch.set_alpha(0.6)
axes[0].axhline(0.5, color='gray', ls='--', lw=1)
axes[0].set(ylabel='AUROC (10-fold CV)',
            title='P300 BCI — comparación de clasificadores\n'
                  f'N={len(y_p3)} ensayos, desbalance 1:{n_nontarget//n_target}')

# Morfología media del ERP por clase
ventanas_ms = [50, 150, 300, 500, 700]   # centros de ventana en ms
for k, (canal_idx, color, lbl) in enumerate([
    (2,  'steelblue', 'Canal Cz (central)'),
    (18, 'tomato',    'Canal Pz (parietal)'),
]):
    axes[1].plot(ventanas_ms,
                  X_target[:, [canal_idx, canal_idx+8, canal_idx+16,
                                canal_idx+24, canal_idx+32]].mean(axis=0),
                  'o-', color=color, lw=2, label=f'{lbl} — Target')
    axes[1].plot(ventanas_ms,
                  X_nontarget[:, [canal_idx, canal_idx+8, canal_idx+16,
                                   canal_idx+24, canal_idx+32]].mean(axis=0),
                  's--', color=color, lw=1.5, alpha=0.5, label=f'{lbl} — Non-target')

axes[1].axvline(300, color='gray', ls=':', lw=1.5, label='P300 (300 ms)')
axes[1].axhline(0,   color='k',    lw=0.8)
axes[1].set(xlabel='Tiempo post-estímulo (ms)', ylabel='Amplitud media (μV)',
            title='Morfología media del ERP\nTarget vs Non-target por canal')
axes[1].legend(fontsize=7.5)

plt.tight_layout()
plt.show()

## Parte 6 — ¿Cuándo usar SVM vs regresión logística?

| Criterio | Regresión Logística | SVM |
|---|---|---|
| **N grande, d moderado** | ✅ Rápida, escala bien | ⚠ Lenta (kernel) |
| **N pequeño, d grande** | ⚠ Puede sobreajustar | ✅ Regularización implícita del margen |
| **Probabilidades calibradas** | ✅ Directo | ⚠ Requiere Platt scaling |
| **Datos no lineales** | ⚠ Necesita features manuales | ✅ Kernel RBF |
| **Interpretabilidad** | ✅ Coeficientes directos | ⚠ Solo coeficientes con kernel lineal |
| **EEG/BCI (N pequeño)** | ✅ Con regularización | ✅ Opción preferida en la literatura |

In [ ]:
# ── Efecto del tamaño muestral en SVM vs LR ───────────────────────────────────
# ¿A partir de cuántos datos convergen ambos modelos?

n_sizes = [20, 40, 80, 150, 250, 400, len(X_tr)]
n_rep   = 8
cv3     = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

modelos_comp = [
    ('LR',      Pipeline([('sc', StandardScaler()),
                           ('clf', LogisticRegression(C=1.0, max_iter=300,
                                    class_weight='balanced'))]),
     'steelblue'),
    ('SVM-RBF', Pipeline([('sc', StandardScaler()),
                           ('clf', SVC(kernel='rbf', C=1.0, gamma='scale',
                                    class_weight='balanced', probability=True))]),
     'tomato'),
]

fig, ax = plt.subplots(figsize=(9, 4))

for nombre, pipe, color in modelos_comp:
    medias, stds = [], []
    for n_s in n_sizes:
        rep_aurocs = []
        for _ in range(n_rep):
            n_p = max(4, round(n_s * y_tr.mean()))
            n_n = n_s - n_p
            ip  = rng.choice(np.where(y_tr==1)[0], n_p, replace=False)
            in_ = rng.choice(np.where(y_tr==0)[0], n_n, replace=False)
            idx_s = np.concatenate([ip, in_])
            if len(np.unique(y_tr[idx_s])) < 2:
                continue
            aucs = cross_val_score(pipe, X_tr[idx_s], y_tr[idx_s],
                                    cv=cv3, scoring='roc_auc')
            rep_aurocs.extend(aucs)
        medias.append(np.mean(rep_aurocs))
        stds.append(np.std(rep_aurocs))

    medias = np.array(medias)
    stds   = np.array(stds)
    ax.plot(n_sizes, medias, 'o-', color=color, lw=2.5, ms=6, label=nombre)
    ax.fill_between(n_sizes, medias-stds, medias+stds, alpha=0.15, color=color)

ax.axhline(0.5, color='gray', ls='--', lw=1, label='Azar')
ax.set(xlabel='Tamaño del conjunto de entrenamiento',
       ylabel='AUROC (3-fold CV)',
       title='SVM-RBF vs Regresión Logística — curvas de aprendizaje\n'
             'Dataset UCI CinC 2012')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('Observación:')
print('  Con N pequeño, SVM-RBF puede superar a LR gracias a la regularización del margen.')
print('  Con N grande, ambos convergen — la diferencia depende principalmente de la')
print('  linealidad del problema y del coste computacional.')

## ✏️ Ejercicios

Los ejercicios usan el dataset **BCI Competition IV Dataset 2a**:
imaginería motora de 4 clases (mano izq., mano der., pies, lengua),
22 canales EEG, 9 sujetos.

> **Fuente:** Brunner, C. et al. (2008). BCI Competition 2008 — Graz data set A.
> Institute for Knowledge Discovery, Graz University of Technology.
> https://www.bbci.de/competition/iv/

1. **Kernel personalizado.** Implementa el kernel de chi-cuadrado
   $k(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \sum_j (x_j - x_j')^2 / (x_j + x_j' + \varepsilon))$
   y evalúalo en el dataset P300 de la Parte 5 usando `SVC(kernel='precomputed')`.
   Compara con el kernel RBF. ¿Cuál es más apropiado para características de potencia espectral?

2. **SVM one-vs-rest para 4 clases.** Implementa un clasificador SVM one-vs-rest para
   imaginería motora de 4 clases usando el dataset BCI IV 2a simulado a continuación.
   Reporta la matriz de confusión y el AUROC one-vs-rest para cada clase.
   ```python
   # Dataset BCI IV 2a simulado (4 clases, 22 características)
   n_cls, n_per = 4, 60
   medias_bci = rng.normal(0, 1, (n_cls, 22))
   X_bci = np.vstack([rng.normal(medias_bci[k], 1.2, (n_per, 22)) for k in range(n_cls)])
   y_bci = np.repeat(np.arange(n_cls), n_per)
   ```

3. **Búsqueda bayesiana de hiperparámetros.** Implementa una búsqueda aleatoria
   (`RandomizedSearchCV`) sobre C ∈ LogUniform(10⁻³, 10³) y
   γ ∈ LogUniform(10⁻⁴, 10¹) con 50 iteraciones. Compara el AUROC obtenido
   con la búsqueda en rejilla de la Parte 4. ¿Cuál encuentra el mejor hiperparámetro?
   ¿Cuánto tiempo tarda cada uno?

4. **Calibración de probabilidades Platt.** El SVM no produce probabilidades
   calibradas por defecto. Usa `CalibratedClassifierCV` con `method='sigmoid'`
   (Platt scaling) y `method='isotonic'` sobre el SVM lineal.
   Grafica el diagrama de calibración (reliability diagram) y calcula el
   Expected Calibration Error (ECE) para el dataset UCI.

5. *(Desafío)* **SVM con LOSO en BCI IV 2a real.** Descarga el dataset BCI Competition
   IV 2a (https://www.bbci.de/competition/iv/). Aplica el pipeline completo:
   filtrado bandpass (8–30 Hz) → CSP (Common Spatial Patterns, 6 filtros) → SVM-RBF
   con búsqueda en rejilla del inner loop → LOSO con 9 sujetos. Reporta exactitud
   por sujeto y exactitud global. Compara con la línea base de permutaciones.

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| PhysioNet CinC 2012 (UCI) | Silva, I. et al. (2012). *Computing in Cardiology*, 39, 245–248. https://physionet.org/content/challenge-2012/ | Dataset principal del Módulo II |
| P300 (diseño experimental) | Blankertz, B. et al. (2011). *NeuroImage*, 56(2), 814–825. https://doi.org/10.1016/j.neuroimage.2010.06.048 | Ejercicios BCI |
| BCI Competition IV 2a | Brunner, C. et al. (2008). https://www.bbci.de/competition/iv/ | Ejercicio de desafío |